In [1]:
# Eve

A = 7
B = 5

p = 11
g = 6

$$A = g^a \bmod p$$

In [2]:
# Brute Force Alice Private key
#   We try every a form 0 to 100 and check is this the Public key

a = range(0, p - 1) # 
a_s: list[int] = []

for a_i in a:
    A_i = pow(g, a_i, p) # Here we try every a
    if A_i == A:
        a_s.append(a_i)

print(f"Found secerts {a_s} for a in {a}")


Found secerts [3] for a in range(0, 10)


## Warum braucht es nur p-1 Versuche?

Bei Diffie-Hellman mit einem Primzahl-Modulus `p` gilt nach dem **Kleinen Satz von Fermat**:

$$g^{p-1} \equiv 1 \pmod{p}$$

Das bedeutet, dass sich die Potenzen von `g` periodisch wiederholen:

$$g^a \equiv g^{a + (p-1)} \pmod{p}$$

Daher sind nur die Exponenten im Bereich `[0, p-2]` (also `p-1` verschiedene Werte) relevant. Alle höheren Exponenten erzeugen die gleichen Ergebnisse wie diese `p-1` Werte.

In diesem Beispiel mit `p = 11`:
- Wir müssen nur `a ∈ {0, 1, 2, 3, 4, 5, 6, 7, 8, 9}` (10 Werte) testen
- `g^10 ≡ 1 (mod 11)`, also wäre `g^11 ≡ g^0`, `g^12 ≡ g^1`, usw.

Das ist auch der Grund, warum im Code steht:
```python
a = range(0, p - 1)  # 0 bis p-2, also p-1 Werte
```

In [3]:
# Brute Force Alice Private key
#   We try every a form 0 to 100 and check is this the Public key

b = range(0, p-1)
b_s: list[int] = []

for b_i in b:
    B_i = pow(g, b_i, p)
    if B_i == B:
        b_s.append(b_i)

print(f"Found secerts {b_s} for b in {b}")

Found secerts [6] for b in range(0, 10)


Es gibt noch weiter möglichkeiten wie 
- Baby-Step Giant-Step
- Pollard-Roh-Methode
- Pohlig-Hellman-Algorithmus
- Index-Calculus-Methode

In [4]:
# Here we calculate with every Private Key the secert key but its always the same
ab = b_s[0] * a_s[0]
sk_f = pow(g, ab, p)

for b in b_s:
    for a in a_s:
        sk = pow(g, (a * b), p)
        assert(sk_f == sk)
        
print(f"The secert key is: {sk_f}")

The secert key is: 4


In [6]:
# Variante 1: Mit Early Exit (schneller für kleines p)
def brute_force_early_exit(public_key: int, g: int, p: int) -> int | None:
    for i in range(p - 1):
        if pow(g, i, p) == public_key:
            return i  # Sofort zurück nach Fund!
    return None

b_optimized = brute_force_early_exit(B, g, p)
print(f"Found b = {b_optimized} with early exit")

# Variante 2: Mit Lookup-Table (O(1) für mehrere Lookups)
def build_lookup_table(g: int, p: int) -> dict[int, int]:
    """Pre-compute alle möglichen Werte"""
    lookup: dict[int, int] = {}
    for i in range(p - 1):
        value = pow(g, i, p)
        if value not in lookup:  # Nur erste Lösung speichern
            lookup[value] = i
    return lookup

# Einmal berechnen
lookup_table = build_lookup_table(g, p)
print(f"Lookup table: {lookup_table}")

# Dann O(1) Lookup für beide Keys
a_fast: int = lookup_table.get(A, -1)
assert a_fast != -1, "A not found in lookup table"
b_fast = lookup_table.get(B, -1)
assert b_fast != -1, "B not found in lookup table"
print(f"Found a = {a_fast}, b = {b_fast} via lookup table")

# Berechne Secret Key
sk_fast = pow(g, a_fast * b_fast, p)
print(f"Secret key: {sk_fast}")

Found b = 6 with early exit
Lookup table: {1: 0, 6: 1, 3: 2, 7: 3, 9: 4, 10: 5, 5: 6, 8: 7, 4: 8, 2: 9}
Found a = 3, b = 6 via lookup table
Secret key: 4
